In [66]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [67]:
file_path = "../data/Processed/furniture_returns_featured.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (5057, 23)


In [68]:
feature_columns = [
    "Product",
    "Value ₹",
    "Transit Days",
    "Return Reason",
    "Condition Hint",
    "Severity",
    "Safety",
    "SLA hrs",
    "Resale Before ₹",
    "Severity Score",
    "Safety Score",
    "Condition Risk Score",
    "Transit Risk Score",
    "SLA Urgency Score"
]

X = df[feature_columns]
y = df["Inspection Outcome"]

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: (5057, 14)
Target shape: (5057,)


In [69]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4045
Testing samples: 1012


In [70]:
import joblib

model_path = "../models/random_forest_inspection_model.pkl"

model_pipeline = joblib.load(model_path)

print("Saved Random Forest model loaded successfully!")

Saved Random Forest model loaded successfully!


In [71]:
y_pred = model_pipeline.predict(X_test)

print("Predictions generated successfully!")
print("Number of predictions:", len(y_pred))

Predictions generated successfully!
Number of predictions: 1012


In [72]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

Model Accuracy: 77.37 %


In [73]:
classification_report_result = classification_report(
    y_test,
    y_pred
)

print(classification_report_result)

              precision    recall  f1-score   support

major damage       0.30      0.47      0.37        49
minor damage       0.95      0.79      0.86       370
   no defect       0.97      0.99      0.98       391
  repairable       0.46      0.47      0.47       135
       scrap       0.19      0.34      0.24        38
     unknown       0.07      0.03      0.05        29

    accuracy                           0.77      1012
   macro avg       0.49      0.52      0.49      1012
weighted avg       0.80      0.77      0.78      1012



In [74]:
report = classification_report(
    y_test,
    y_pred,
    output_dict=True
)

metrics_df = pd.DataFrame(report).transpose()

metrics_df.round(3)

,precision,recall,f1-score,support
major damage,0.299,0.469,0.365,49.000
minor damage,0.945,0.792,0.862,370.000
no defect,0.968,0.995,0.981,391.000
repairable,0.464,0.474,0.469,135.000
scrap,0.186,0.342,0.241,38.000
unknown,0.067,0.034,0.045,29.000
accuracy,0.774,0.774,0.774,0.774
macro avg,0.488,0.518,0.494,1012.000
weighted avg,0.805,0.774,0.785,1012.000


In [75]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=model_pipeline.classes_
)

print("Class order:")
print(list(model_pipeline.classes_))

print("\nConfusion Matrix:")
print(cm)

Class order:
['major damage', 'minor damage', 'no defect', 'repairable', 'scrap', 'unknown']

Confusion Matrix:
[[ 23   0   0   1  25   0]
 [  4 293   1  64   1   7]
 [  0   0 389   0   0   2]
 [ 28   9   0  64  30   4]
 [ 21   0   0   3  13   1]
 [  1   8  12   6   1   1]]


In [76]:
confusion_matrix_df = pd.DataFrame(
    cm,
    index=model_pipeline.classes_,
    columns=model_pipeline.classes_
)

confusion_matrix_df

,major damage,minor damage,no defect,repairable,scrap,unknown
major damage,23,0,0,1,25,0
minor damage,4,293,1,64,1,7
no defect,0,0,389,0,0,2
repairable,28,9,0,64,30,4
scrap,21,0,0,3,13,1
unknown,1,8,12,6,1,1


In [77]:
major_damage_actual = y_test == "major damage"

major_damage_false_negatives = (
    major_damage_actual &
    (y_pred != "major damage")
)

major_damage_fn_count = major_damage_false_negatives.sum()

print(
    "Major damage false negatives:",
    major_damage_fn_count
)

Major damage false negatives: 26


In [78]:
scrap_actual = y_test == "scrap"

scrap_false_negatives = (
    scrap_actual &
    (y_pred != "scrap")
)

scrap_fn_count = scrap_false_negatives.sum()

print(
    "Scrap false negatives:",
    scrap_fn_count
)

Scrap false negatives: 25


In [79]:
major_damage_recall = recall_score(
    y_test,
    y_pred,
    labels=["major damage"],
    average=None
)[0]

scrap_recall = recall_score(
    y_test,
    y_pred,
    labels=["scrap"],
    average=None
)[0]

print(
    "Major Damage Recall:",
    round(major_damage_recall * 100, 2),
    "%"
)

print(
    "Scrap Recall:",
    round(scrap_recall * 100, 2),
    "%"
)

Major Damage Recall: 46.94 %
Scrap Recall: 34.21 %


In [80]:
macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print(
    "Macro F1 Score:",
    round(macro_f1, 3)
)

print(
    "Weighted F1 Score:",
    round(weighted_f1, 3)
)

Macro F1 Score: 0.494
Weighted F1 Score: 0.785


In [81]:
error_costs = {
    "major damage": 5,
    "scrap": 5,
    "repairable": 3,
    "minor damage": 2,
    "no defect": 1,
    "unknown": 1
}

print("Illustrative error costs:")
print(error_costs)

Illustrative error costs:
{'major damage': 5, 'scrap': 5, 'repairable': 3, 'minor damage': 2, 'no defect': 1, 'unknown': 1}


In [82]:
total_error_cost = 0

for actual, predicted in zip(y_test, y_pred):
    if actual != predicted:
        total_error_cost += error_costs.get(actual, 1)

print(
    "Total illustrative error cost:",
    total_error_cost
)

Total illustrative error cost: 652


In [83]:
error_count = np.sum(
    y_test.values != y_pred
)

average_error_cost = (
    total_error_cost / error_count
    if error_count > 0
    else 0
)

print(
    "Average illustrative cost per error:",
    round(average_error_cost, 2)
)

Average illustrative cost per error: 2.85


In [84]:
error_analysis = X_test.copy()

error_analysis["Actual Outcome"] = y_test
error_analysis["Predicted Outcome"] = y_pred

error_analysis["Correct Prediction"] = (
    error_analysis["Actual Outcome"] ==
    error_analysis["Predicted Outcome"]
)

error_analysis["Error Cost"] = (
    error_analysis["Actual Outcome"]
    .map(error_costs)
    .fillna(1)
)

error_analysis.head(10)

,Product,Value ₹,Transit Days,Return Reason,Condition Hint,Severity,Safety,SLA hrs,Resale Before ₹,Severity Score,Safety Score,Condition Risk Score,Transit Risk Score,SLA Urgency Score,Actual Outcome,Predicted Outcome,Correct Prediction,Error Cost
952,sofa,38722,6,damaged,structural crack,low,medium,24,35299,1,2,3,3,3,minor damage,minor damage,True,2
3558,bed,26326,2,damaged,torn fabric,high,low,24,30385,3,1,2,1,3,scrap,scrap,True,5
4968,bookshelf,6931,3,damaged,chipped surface,medium,low,48,6459,2,1,1,2,2,minor damage,repairable,False,2
1078,bookshelf,11281,1,damaged,broken shelf,high,high,24,10672,3,3,3,1,3,scrap,scrap,True,5
4981,chair,6569,0,size issue,no visible damage,unknown,low,48,5777,1,1,1,1,2,unknown,no defect,False,1
3623,bookshelf,10212,4,defective,door misalignment,medium,low,72,21997,2,1,2,2,1,repairable,repairable,True,3
905,bookshelf,26326,0,defective,loose joints,medium,high,72,21997,2,3,2,1,1,unknown,repairable,False,1
2614,wardrobe,29416,5,wrong product,no visible damage,unknown,low,48,25176,1,1,1,2,2,no defect,no defect,True,1
747,chair,26326,1,damaged,scratched surface,high,low,48,6455,3,1,1,1,2,repairable,scrap,False,3
1557,bed,26326,2,unknown,wrong fabric,low,low,24,37743,1,1,1,1,3,minor damage,minor damage,True,2


In [85]:
incorrect_predictions = error_analysis[
    error_analysis["Correct Prediction"] == False
]

print(
    "Total incorrect predictions:",
    len(incorrect_predictions)
)

incorrect_predictions.head(20)

Total incorrect predictions: 229


,Product,Value ₹,Transit Days,Return Reason,Condition Hint,Severity,Safety,SLA hrs,Resale Before ₹,Severity Score,Safety Score,Condition Risk Score,Transit Risk Score,SLA Urgency Score,Actual Outcome,Predicted Outcome,Correct Prediction,Error Cost
4968,bookshelf,6931,3,damaged,chipped surface,medium,low,48,6459,2,1,1,2,2,minor damage,repairable,False,2
4981,chair,6569,0,size issue,no visible damage,unknown,low,48,5777,1,1,1,1,2,unknown,no defect,False,1
905,bookshelf,26326,0,defective,loose joints,medium,high,72,21997,2,3,2,1,1,unknown,repairable,False,1
747,chair,26326,1,damaged,scratched surface,high,low,48,6455,3,1,1,1,2,repairable,scrap,False,3
1699,coffee table,14450,5,damaged,scratched surface,high,medium,72,12074,3,2,1,2,1,scrap,major damage,False,5
2917,chair,6204,0,defective,cushion damage,low,high,48,5871,1,3,2,1,2,minor damage,unknown,False,2
4449,sofa,34918,2,defective,cushion damage,high,medium,72,31213,3,2,2,1,1,repairable,scrap,False,3
4572,dining table,26326,1,damaged,stained fabric,high,low,72,38954,3,1,1,1,1,repairable,scrap,False,3
769,chair,7143,0,not as expected,minor scratch,low,low,72,21997,1,1,1,1,1,unknown,minor damage,False,1
1134,dining table,35300,0,damaged,cracked wood,high,high,48,31738,3,3,3,1,2,major damage,scrap,False,5


In [86]:
error_pairs = (
    incorrect_predictions
    .groupby(
        ["Actual Outcome", "Predicted Outcome"]
    )
    .size()
    .reset_index(name="Count")
    .sort_values(
        "Count",
        ascending=False
    )
)

error_pairs

,Actual Outcome,Predicted Outcome,Count
4,minor damage,repairable,64
10,repairable,scrap,30
8,repairable,major damage,28
1,major damage,scrap,25
12,scrap,major damage,21
17,unknown,no defect,12
9,repairable,minor damage,9
16,unknown,minor damage,8
6,minor damage,unknown,7
18,unknown,repairable,6


In [87]:
major_damage_errors = error_analysis[
    (error_analysis["Actual Outcome"] == "major damage") &
    (error_analysis["Predicted Outcome"] != "major damage")
]

print(
    "Major damage missed by model:",
    len(major_damage_errors)
)

major_damage_errors.head(20)

Major damage missed by model: 26


,Product,Value ₹,Transit Days,Return Reason,Condition Hint,Severity,Safety,SLA hrs,Resale Before ₹,Severity Score,Safety Score,Condition Risk Score,Transit Risk Score,SLA Urgency Score,Actual Outcome,Predicted Outcome,Correct Prediction,Error Cost
1134,dining table,35300,0,damaged,cracked wood,high,high,48,31738,3,3,3,1,2,major damage,scrap,False,5
1009,bookshelf,11059,1,damaged,chipped surface,high,high,48,10418,3,3,1,1,2,major damage,scrap,False,5
2319,dining table,35605,5,defective,broken frame,high,high,48,29541,3,3,3,2,2,major damage,scrap,False,5
4284,bed,45516,2,damaged,torn fabric,high,high,48,38608,3,3,2,1,2,major damage,scrap,False,5
1269,chair,26326,2,defective,door misalignment,high,medium,48,4850,3,2,2,1,2,major damage,scrap,False,5
50,chair,3604,0,defective,door misalignment,high,medium,72,2718,3,2,2,1,1,major damage,scrap,False,5
2076,chair,6190,2,damaged,torn fabric,high,high,72,5745,3,3,2,1,1,major damage,scrap,False,5
1088,coffee table,8186,4,defective,cushion damage,high,unknown,24,6143,3,1,2,2,3,major damage,scrap,False,5
1842,wardrobe,28579,4,damaged,structural crack,high,low,24,25190,3,1,3,2,3,major damage,scrap,False,5
3264,bed,37573,9,damaged,scratched surface,high,high,72,34555,3,3,1,3,1,major damage,scrap,False,5


In [88]:
scrap_errors = error_analysis[
    (error_analysis["Actual Outcome"] == "scrap") &
    (error_analysis["Predicted Outcome"] != "scrap")
]

print(
    "Scrap cases missed by model:",
    len(scrap_errors)
)

scrap_errors.head(20)

Scrap cases missed by model: 25


,Product,Value ₹,Transit Days,Return Reason,Condition Hint,Severity,Safety,SLA hrs,Resale Before ₹,Severity Score,Safety Score,Condition Risk Score,Transit Risk Score,SLA Urgency Score,Actual Outcome,Predicted Outcome,Correct Prediction,Error Cost
1699,coffee table,14450,5,damaged,scratched surface,high,medium,72,12074,3,2,1,2,1,scrap,major damage,False,5
3650,bed,43674,2,damaged,structural crack,high,medium,48,41080,3,2,3,1,2,scrap,major damage,False,5
1493,coffee table,14081,1,defective,door misalignment,high,high,24,12913,3,3,2,1,3,scrap,major damage,False,5
1762,chair,7524,1,damaged,broken leg,high,medium,72,6113,3,2,3,1,1,scrap,unknown,False,5
3653,chair,7419,0,defective,unknown,high,medium,24,6167,3,2,1,1,3,scrap,major damage,False,5
3029,bookshelf,11001,0,defective,faulty hinge,high,high,48,10101,3,3,2,1,2,scrap,major damage,False,5
370,bookshelf,11653,0,defective,broken frame,high,low,48,10876,3,1,3,1,2,scrap,repairable,False,5
4915,bed,50193,4,damaged,pest damage,high,low,48,43445,3,1,2,2,2,scrap,major damage,False,5
616,wardrobe,26783,1,defective,cushion damage,high,medium,48,24363,3,2,2,1,2,scrap,major damage,False,5
1534,coffee table,12932,0,defective,door misalignment,high,high,24,11488,3,3,2,1,3,scrap,major damage,False,5


In [89]:
prediction_probabilities = (
    model_pipeline.predict_proba(X_test)
)

classes = model_pipeline.classes_

probability_df = pd.DataFrame(
    prediction_probabilities,
    columns=[
        f"Probability_{class_name}"
        for class_name in classes
    ]
)

probability_df.head()

,Probability_major damage,Probability_minor damage,Probability_no defect,Probability_repairable,Probability_scrap,Probability_unknown
0,0.003357,0.821383,0.000000,0.017999,0.028867,0.128394
1,0.376203,0.002201,0.000061,0.172503,0.418192,0.030839
2,0.010670,0.416892,0.009583,0.443076,0.005313,0.114465
3,0.226070,0.000227,0.000000,0.164337,0.558711,0.050654
4,0.000308,0.000446,0.844654,0.000168,0.000000,0.154423


In [90]:
error_analysis["ML Confidence"] = (
    prediction_probabilities.max(axis=1) * 100
).round(2)

error_analysis[
    [
        "Actual Outcome",
        "Predicted Outcome",
        "ML Confidence",
        "Correct Prediction"
    ]
].head(20)

,Actual Outcome,Predicted Outcome,ML Confidence,Correct Prediction
952,minor damage,minor damage,82.14,True
3558,scrap,scrap,41.82,True
4968,minor damage,repairable,44.31,False
1078,scrap,scrap,55.87,True
4981,unknown,no defect,84.47,False
3623,repairable,repairable,57.93,True
905,unknown,repairable,69.90,False
2614,no defect,no defect,86.28,True
747,repairable,scrap,39.26,False
1557,minor damage,minor damage,91.84,True


In [91]:
low_confidence_predictions = error_analysis[
    error_analysis["ML Confidence"] < 60
]

print(
    "Predictions with confidence below 60%:",
    len(low_confidence_predictions)
)

low_confidence_predictions.head(20)

Predictions with confidence below 60%: 253


,Product,Value ₹,Transit Days,Return Reason,Condition Hint,Severity,Safety,SLA hrs,Resale Before ₹,Severity Score,Safety Score,Condition Risk Score,Transit Risk Score,SLA Urgency Score,Actual Outcome,Predicted Outcome,Correct Prediction,Error Cost,ML Confidence
3558,bed,26326,2,damaged,torn fabric,high,low,24,30385,3,1,2,1,3,scrap,scrap,True,5,41.82
4968,bookshelf,6931,3,damaged,chipped surface,medium,low,48,6459,2,1,1,2,2,minor damage,repairable,False,2,44.31
1078,bookshelf,11281,1,damaged,broken shelf,high,high,24,10672,3,3,3,1,3,scrap,scrap,True,5,55.87
3623,bookshelf,10212,4,defective,door misalignment,medium,low,72,21997,2,1,2,2,1,repairable,repairable,True,3,57.93
747,chair,26326,1,damaged,scratched surface,high,low,48,6455,3,1,1,1,2,repairable,scrap,False,3,39.26
1699,coffee table,14450,5,damaged,scratched surface,high,medium,72,12074,3,2,1,2,1,scrap,major damage,False,5,37.46
2917,chair,6204,0,defective,cushion damage,low,high,48,5871,1,3,2,1,2,minor damage,unknown,False,2,58.68
4449,sofa,34918,2,defective,cushion damage,high,medium,72,31213,3,2,2,1,1,repairable,scrap,False,3,37.04
716,bed,53459,2,damaged,broken frame,medium,low,48,44711,2,1,3,1,2,repairable,repairable,True,3,51.87
4572,dining table,26326,1,damaged,stained fabric,high,low,72,38954,3,1,1,1,1,repairable,scrap,False,3,40.03


In [92]:
model = model_pipeline.named_steps["model"]

feature_names = (
    model_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        by="Importance",
        ascending=False
    )
)

feature_importance.head(15)

,Feature,Importance
47,numerical__Severity Score,0.086534
46,numerical__Resale Before ₹,0.080193
26,categorical__Condition Hint_no visible damage,0.080131
43,numerical__Value ₹,0.078304
35,categorical__Severity_high,0.072033
36,categorical__Severity_low,0.066519
38,categorical__Severity_unknown,0.060378
37,categorical__Severity_medium,0.048711
44,numerical__Transit Days,0.047753
49,numerical__Condition Risk Score,0.038027


In [93]:
print("=" * 60)
print("MODEL EVALUATION SUMMARY")
print("=" * 60)

print("\nModel:")
print("Random Forest Classifier")

print("\nDataset:")
print("Furniture Return Prioritizer")

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print("\nAccuracy:")
print(round(accuracy * 100, 2), "%")

print("\nMacro F1:")
print(round(macro_f1, 3))

print("\nWeighted F1:")
print(round(weighted_f1, 3))

print("\nMajor Damage False Negatives:")
print(major_damage_fn_count)

print("\nScrap False Negatives:")
print(scrap_fn_count)

print("\nLow Confidence Predictions:")
print(len(low_confidence_predictions))

print("\nTop 10 Features:")
print(
    feature_importance
    .head(10)
    .to_string(index=False)
)

MODEL EVALUATION SUMMARY

Model:
Random Forest Classifier

Dataset:
Furniture Return Prioritizer

Training Samples: 4045
Testing Samples: 1012

Accuracy:
77.37 %

Macro F1:
0.494

Weighted F1:
0.785

Major Damage False Negatives:
26

Scrap False Negatives:
25

Low Confidence Predictions:
253

Top 10 Features:
                                      Feature  Importance
                    numerical__Severity Score    0.086534
                   numerical__Resale Before ₹    0.080193
categorical__Condition Hint_no visible damage    0.080131
                           numerical__Value ₹    0.078304
                   categorical__Severity_high    0.072033
                    categorical__Severity_low    0.066519
                categorical__Severity_unknown    0.060378
                 categorical__Severity_medium    0.048711
                      numerical__Transit Days    0.047753
              numerical__Condition Risk Score    0.038027


In [94]:
evaluation_output_path = (
    "../data/Processed/"
    "model_evaluation_results.xlsx"
)

with pd.ExcelWriter(
    evaluation_output_path,
    engine="openpyxl"
) as writer:

    metrics_df.to_excel(
        writer,
        sheet_name="Classification Metrics"
    )

    confusion_matrix_df.to_excel(
        writer,
        sheet_name="Confusion Matrix"
    )

    error_pairs.to_excel(
        writer,
        sheet_name="Error Analysis",
        index=False
    )

    feature_importance.head(20).to_excel(
        writer,
        sheet_name="Feature Importance",
        index=False
    )

print("Evaluation results saved successfully!")
print(evaluation_output_path)

Evaluation results saved successfully!
../data/Processed/model_evaluation_results.xlsx


In [95]:
print("=" * 60)
print("NOTEBOOK 06 COMPLETED")
print("=" * 60)

print("\nKey Evaluation Measures:")
print("- Accuracy")
print("- Precision")
print("- Recall")
print("- F1 Score")
print("- Confusion Matrix")
print("- Major Damage False Negatives")
print("- Scrap False Negatives")
print("- Cost-Weighted Error Analysis")
print("- ML Confidence")
print("- Feature Importance")

print("\nImportant Note:")
print(
    "The model uses pre-inspection features only. "
    "Post-inspection value-loss fields were excluded "
    "to avoid data leakage."
)

NOTEBOOK 06 COMPLETED

Key Evaluation Measures:
- Accuracy
- Precision
- Recall
- F1 Score
- Confusion Matrix
- Major Damage False Negatives
- Scrap False Negatives
- Cost-Weighted Error Analysis
- ML Confidence
- Feature Importance

Important Note:
The model uses pre-inspection features only. Post-inspection value-loss fields were excluded to avoid data leakage.
